# Phase 0 — Data Acquisition & Dataset Preparation
### Project: Predicting Protein Thermostability from Sequence Features
**Author:** Mauli Bhavsar  
**Date:** April 2026  

---

## Overview

This notebook documents the complete data acquisition pipeline — from 
identifying a suitable dataset to downloading it, understanding its 
structure, cleaning it, and preparing it for machine learning.

Before any machine learning can happen, we need to answer three questions:

1. **What data do we need?** — Protein sequences with experimentally 
   measured thermal stability values
2. **Where do we get it?** — The Meltome Atlas, a large publicly available 
   thermostability dataset
3. **How do we prepare it?** — Parse the raw files, assign binary labels, 
   and save a clean CSV

This notebook answers all three questions in detail.

---

## 1. Why This Dataset?

### The Meltome Atlas

The dataset we use comes from a landmark 2020 paper:

> Jarzab et al. (2020). Meltome atlas — thermal proteome stability 
> across the tree of life. *Nature Methods*, 17, 495–503.

The researchers used a technique called **Thermal Proteome Profiling (TPP)** 
to measure the melting temperature (Tm) of thousands of proteins across 
13 different organisms — from bacteria to humans.

**Why is this dataset ideal for our ML project?**

| Reason | Details |
|--------|---------|
| Size | 201,283 proteins — large enough for robust ML |
| Quality | Experimentally measured Tm values — not predicted |
| Diversity | Proteins from 13 organisms across the tree of life |
| Publicly available | Freely downloadable — reproducible research |
| Used in published ML benchmarks | Validated by the research community |

### What is Melting Temperature (Tm)?

When you heat a protein, it gradually unfolds — losing its 3D structure 
and therefore its function. The **melting temperature (Tm)** is the 
temperature at which exactly 50% of the protein population is unfolded.

- A protein with **high Tm (e.g. 80°C)** survives high temperatures 
  without unfolding → **Thermostable**
- A protein with **low Tm (e.g. 35°C)** unfolds at normal temperatures 
  → **Mesostable**

Thermostable proteins are found in organisms called **thermophiles** — 
microorganisms that live in extreme heat environments like hot springs 
and deep sea hydrothermal vents. A famous example is *Thermus aquaticus*, 
the bacterium from Yellowstone hot springs whose DNA polymerase enzyme 
(Taq polymerase) is used in every PCR reaction in every biology lab 
in the world.

## 2. How Was the Dataset Downloaded?

The Meltome Atlas data is hosted on the FLIP GitHub repository:

> Dallago et al. (2022). FLIP: Benchmark tasks in fitness landscape 
> inference for proteins. *Cell Systems*.  
> GitHub: https://github.com/J-SNACKKB/FLIP

### Files Downloaded

We downloaded two files manually from the GitHub repository:

**File 1: `full_dataset_sequences.fasta`**  
Contains the amino acid sequence of every protein along with its 
measured melting temperature in the header line.

**File 2: `full_dataset.json`**  
Contains additional metadata for each protein including UniProt 
accession numbers and organism information.

### How to Download

1. Go to: https://github.com/J-SNACKKB/FLIP
2. Navigate to: `splits` → `meltome`
3. Download `full_dataset_sequences.fasta.zip` and `full_dataset.json.zip`
4. Unzip both files into `data/raw/`

```bash
cd ~/protein_thermo_ml/data/raw
unzip full_dataset_sequences.fasta.zip
unzip full_dataset.json.zip
```

### Why Did We Only Use the FASTA File?

When we tried to merge the FASTA and JSON files using `protein_id` as 
the common key, we found the ID formats were completely different:

| File | ID format | Example |
|------|-----------|---------|
| FASTA | `UniprotAccession_OrganismName` | `A0A023T4K3_Caenorhabditis_elegans_lysate` |
| JSON | `UniprotAccession_GeneName` | `A0A023PXQ4_YMR173W-A` |

Since the FASTA file already contains **both** the sequence and the 
melting temperature in the header line, we have everything we need 
without the JSON file. The JSON was therefore not used.

## 3. Understanding the Raw FASTA File

A **FASTA file** is the most common file format for storing biological 
sequences. Every entry has exactly two parts:

**The header line** — starts with `>` and contains the protein ID and 
any additional information:

A0A023T4K3_Caenorhabditis_elegans_lysate MELTING_POINT=37.9629473421417

**The sequence lines** — the actual amino acid sequence in single-letter 
code continuing on the next lines: 

MSGEEEKAADFYVRYYVGHKGKFGHEFLEFEFRPNGSLRYANNSNYKNDTMIRKEATVSE
SVLSELKRIIEDSEIMQEDDDNWPEPDKIGRQELEILYKNEHISFTTGKIGALADVNNSK

### Single Letter Amino Acid Codes

Each letter represents one amino acid:

| Letter | Amino Acid | Letter | Amino Acid |
|--------|-----------|--------|-----------|
| A | Alanine | M | Methionine |
| C | Cysteine | N | Asparagine |
| D | Aspartate | P | Proline |
| E | Glutamate | Q | Glutamine |
| F | Phenylalanine | R | Arginine |
| G | Glycine | S | Serine |
| H | Histidine | T | Threonine |
| I | Isoleucine | V | Valine |
| K | Lysine | W | Tryptophan |
| L | Leucine | Y | Tyrosine |

So the sequence `MKKQTLSEW...` means the protein starts with 
Methionine → Lysine → Lysine → Glutamine → Threonine → ...

In [2]:
# Let's peek at the first entry of the raw FASTA file
# to see exactly what we are working with

# open() opens a file for reading
# we read only the first 5 lines using a counter
print("=== First entry in the raw FASTA file ===\n")

line_count = 0
with open('../data/raw/full_dataset_sequences.fasta', 'r') as f:
    for line in f:
        print(line.strip())  # .strip() removes newline characters
        line_count += 1
        if line_count == 5:  # stop after 5 lines
            break

print("\n...")
print("\n=== Explanation ===")
print("Line 1: Header line starting with '>'")
print("        Contains protein ID and MELTING_POINT value")
print("Lines 2-4: Amino acid sequence split across multiple lines")
print("           (FASTA files wrap sequences at 60-80 characters per line)")

=== First entry in the raw FASTA file ===

>A0A023T4K3_Caenorhabditis_elegans_lysate MELTING_POINT=37.9629473421417
MSGEEEKAADFYVRYYVGHKGKFGHEFLEFEFRPNGSLRYANNSNYKNDTMIRKEATVSE
SVLSELKRIIEDSEIMQEDDDNWPEPDKIGRQELEILYKNEHISFTTGKIGALADVNNSK
DPDGLRSFYYLVQDLKCLVFSLIGLHFKIKPI
>A0A023T778_Mus_musculus_BMDC_lysate MELTING_POINT=54.4253424806097

...

=== Explanation ===
Line 1: Header line starting with '>'
        Contains protein ID and MELTING_POINT value
Lines 2-4: Amino acid sequence split across multiple lines
           (FASTA files wrap sequences at 60-80 characters per line)


### What We Can See

The first protein in the file is:
- **ID:** `A0A023T4K3_Caenorhabditis_elegans_lysate`
  - `A0A023T4K3` = UniProt accession number (unique protein identifier)
  - `Caenorhabditis_elegans_lysate` = organism (*C. elegans* is a small 
    roundworm commonly used in biology research)
- **Melting point:** 37.96°C — this is a mesostable protein (Tm < 45°C)
  so it will get **label = 0** in our dataset
- **Sequence:** `MSGEEEKAADFYVRYYVGHKGKFG...` — 152 amino acids long

The second protein shown is from *Mus musculus* (mouse) with 
Tm = 54.4°C — this falls in our **grey zone (45–60°C)** and will 
be **dropped** from our dataset.

This perfectly illustrates why we needed to parse and filter the data 
carefully before any ML work.

## 4. How We Parsed and Cleaned the Data

The parsing is handled by `src/01_parse_data.py`. Here we walk through 
every step of that script in detail.

### Step 1 — Reading the FASTA file with Biopython

We use **Biopython's SeqIO module** to read the FASTA file. Biopython is 
a Python library specifically built for bioinformatics tasks — it handles 
all the complexity of biological file formats so we don't have to write 
our own parser.

```python
from Bio import SeqIO

for record in SeqIO.parse("full_dataset_sequences.fasta", "fasta"):
    print(record.id)          # protein ID
    print(str(record.seq))    # amino acid sequence
    print(record.description) # full header line including MELTING_POINT
```

`SeqIO.parse()` works like a loop — it reads one protein at a time from 
the file, which is memory efficient for large files like ours (201,283 proteins).

### Step 2 — Extracting the Melting Point

The melting point is embedded in the header line as `MELTING_POINT=37.96`. 
We extract it by:
1. Splitting the header line by spaces → `["A0A023T4K3_...", "MELTING_POINT=37.96"]`
2. Finding the part that starts with `"MELTING_POINT="`
3. Splitting by `"="` and taking the second part → `"37.96"`
4. Converting to a float (decimal number) → `37.96`

### Step 3 — Assigning Binary Labels

This is the most important design decision in the entire project.

We convert the continuous Tm value into a binary label:

| Tm range | Label | Reasoning |
|----------|-------|-----------|
| Tm > 60°C | 1 (Thermostable) | Clearly thermostable — proteins from thermophilic organisms |
| Tm < 45°C | 0 (Mesostable) | Clearly mesostable — proteins from normal temperature organisms |
| 45°C ≤ Tm ≤ 60°C | Dropped | Biologically ambiguous — cannot confidently assign either label |

**Why 60°C and 45°C specifically?**

These thresholds are not arbitrary. They come from the biology:
- 60°C is the commonly accepted lower bound for thermophilic organisms
- 45°C is the upper bound for mesophilic organisms
- The gap between them is the grey zone where proteins could belong 
  to either category — including them would introduce **label noise** 
  which would hurt model performance

**Why binary classification instead of regression?**

We could have kept Tm as a continuous value and predicted the exact 
temperature — this is called **regression**. We chose binary 
classification instead because:
1. It is a cleaner learning problem for a first ML project
2. Most real-world decisions are binary — "will this enzyme survive at 65°C?"
3. It allows us to use **AUROC** as our evaluation metric, which handles 
   class imbalance better than regression metrics like RMSE

In [6]:
import subprocess

# Run the script using its absolute path
result = subprocess.run(
    ['python3', '/home/maulibhavsar/protein_thermo_ml/src/01_parse_data.py'],
    capture_output=True,
    text=True,
    cwd='/home/maulibhavsar/protein_thermo_ml'  # project root
)

print(result.stdout)

if result.returncode != 0:
    print("ERRORS:")
    print(result.stderr)

Parsing FASTA file...
  Total records parsed: 201,283
  Tm range: 27.6 – 99.0 °C

After labelling and dropping grey zone:
  Thermostable (label=1): 20,051
  Mesostable   (label=0): 16,049
  Grey zone dropped     : 165,183
  Total kept            : 36,100

Saved to: data/processed/02_dataset_clean.csv
Phase 1 complete!



### Output Explained

| Statistic | Value | Meaning |
|-----------|-------|---------|
| Total records parsed | 201,283 | All proteins in the Meltome Atlas |
| Tm range | 27.6 – 99.0°C | Realistic biological range |
| Thermostable (label=1) | 20,051 | Proteins with Tm > 60°C |
| Mesostable (label=0) | 16,049 | Proteins with Tm < 45°C |
| Grey zone dropped | 165,183 | Proteins with 45°C ≤ Tm ≤ 60°C |
| Total kept | 36,100 | Final dataset size for ML |

The majority of proteins (165,183 out of 201,283 — about 82%) fall in 
the grey zone and are dropped. This makes biological sense — most proteins 
from normal organisms operate at temperatures between 45–60°C. The clearly 
thermostable and clearly mesostable proteins are the minority, but we still 
have a very healthy 36,100 samples for machine learning.

## 5. The Final Clean Dataset

After parsing and labelling, we have a clean CSV file saved at:
`data/processed/02_dataset_clean.csv`

Let's load it and take a final look to confirm everything is correct.

In [7]:
import pandas as pd

# Load the clean dataset
df = pd.read_csv('/home/maulibhavsar/protein_thermo_ml/data/processed/02_dataset_clean.csv')

# Basic inspection
print("=== Final Clean Dataset ===")
print(f"Shape         : {df.shape}")
print(f"Columns       : {list(df.columns)}")
print(f"\nClass Distribution:")
print(f"  Thermostable (label=1): {(df.label==1).sum():,} ({(df.label==1).mean()*100:.1f}%)")
print(f"  Mesostable   (label=0): {(df.label==0).sum():,} ({(df.label==0).mean()*100:.1f}%)")
print(f"\nTm Statistics:")
print(f"  Min : {df.tm.min():.1f}°C")
print(f"  Max : {df.tm.max():.1f}°C")
print(f"  Mean: {df.tm.mean():.1f}°C")
print(f"\nFirst 3 rows:")
# Show first 3 rows but truncate sequence to first 30 characters
# so it fits nicely on screen
df_display = df.head(3).copy()
df_display['sequence'] = df_display['sequence'].str[:30] + '...'
print(df_display.to_string(index=False))

=== Final Clean Dataset ===
Shape         : (36100, 4)
Columns       : ['protein_id', 'sequence', 'tm', 'label']

Class Distribution:
  Thermostable (label=1): 20,051 (55.5%)
  Mesostable   (label=0): 16,049 (44.5%)

Tm Statistics:
  Min : 27.6°C
  Max : 99.0°C
  Mean: 55.4°C

First 3 rows:
                                            protein_id                          sequence        tm  label
A0A0K2H409_Geobacillus_stearothermophilus_NCA26_lysate MKKQTLSEWIEQLRQDPNVVYWHEIEPKEA... 70.937807      1
A0A0K2H416_Geobacillus_stearothermophilus_NCA26_lysate MKLYDSILDLIGGTPIVKLRRLPDPNGAEV... 72.330714      1
A0A0K2H440_Geobacillus_stearothermophilus_NCA26_lysate MVYKALTIAGSDSGGGAGIQADLKTFQELG... 62.177204      1


## 6. Summary

This notebook documented the complete data acquisition pipeline for our 
protein thermostability ML project.

### What We Did

| Step | Action | Result |
|------|--------|--------|
| 1 | Identified the Meltome Atlas as our dataset | Chose a large, experimentally validated, publicly available thermostability database |
| 2 | Downloaded FASTA and JSON files from FLIP GitHub | Two raw files saved in `data/raw/` |
| 3 | Understood the FASTA file structure | Each entry has a header with protein ID + Tm, and sequence lines below |
| 4 | Parsed sequences and Tm values using Biopython | 201,283 proteins extracted into a pandas DataFrame |
| 5 | Assigned binary labels using Tm thresholds | Tm > 60°C → label 1, Tm < 45°C → label 0 |
| 6 | Dropped the grey zone (45–60°C) | 165,183 ambiguous proteins removed |
| 7 | Saved clean labelled dataset | `data/processed/02_dataset_clean.csv` — 36,100 proteins ready for ML |

### Key Decisions Made

| Decision | Choice | Reason |
|----------|--------|--------|
| Dataset | Meltome Atlas | Large, experimentally measured, publicly available |
| Problem type | Binary classification | Cleaner than regression, biologically meaningful |
| Thermostable cutoff | Tm > 60°C | Lower bound for thermophilic organisms |
| Mesostable cutoff | Tm < 45°C | Upper bound for mesophilic organisms |
| Grey zone | Dropped | Biologically ambiguous — would introduce label noise |

The clean dataset of 36,100 labelled proteins is now ready. 
Proceed to `01_data_parsing.ipynb` for detailed inspection and 
visualisation of this dataset.
